In [1]:
import boto3
import time
import argparse
import sys
import json
from datetime import datetime
from typing import List, Dict, Optional

In [14]:
!pip install thrift_sasl

Defaulting to user installation because normal site-packages is not writeable
  Preparing metadata (setup.py) ... done
Using legacy 'setup.py install' for pure-sasl, since package 'wheel' is not installed.
    Running setup.py install for pure-sasl ... done


In [1]:
# from pyhive import hive
# import pandas as pd

# # HiveServer2 连接参数
# host = "10.0.21.126"  # HiveServer2 主机地址
# port = 10000         # HiveServer2 端口
# username = "root"    # 用户名
# database = "default" # 指定数据库

# # 建立连接
# conn = hive.connect(
#     host=host,
#     port=port,
#     username=username,
#     database=database
#     #auth='NOSASL'  # 适用于无认证模式
# )

# # 创建游标
# cursor = conn.cursor()

# # 执行查询
# query = "SELECT * FROM poker. ods_temp_test_tab limit 10"
# cursor.execute(query)

# # 获取数据并转换为 DataFrame
# column_names = [desc[0] for desc in cursor.description]
# rows = cursor.fetchall()
# df = pd.DataFrame(rows, columns=column_names)

# # 关闭连接
# cursor.close()
# conn.close()

# # 显示数据
# print(df)

In [7]:
import json
import hmac
import hashlib
import base64
import requests

# ===== 配置项 =====
LAMBDA_URL = "https://sptadsyzvugpstxq5t5mfc54yq0aifuh.lambda-url.ap-southeast-1.on.aws/"  # 替换为你的 API Gateway URL
SECRET_KEY = b"3T94VuAmaGyRmecrXst"  # 必须与 Lambda 中的 SECRET_KEY 一致（bytes）

# 构造 QueryData 内容（模拟一局德州扑克）
query_data = {
    "szToken": "test_token_123",
    "round": 1,
    "gameType": 450,
    "smallblind": 10,
    "users": [
        {"nPlayerId": 144365, "seat": 0, "score": 1000, "bRobot": 0},   # 真人（目标用户）
        {"nPlayerId": 999999,  "seat": 1, "score": 800,  "bRobot": 1},   # 机器人
        {"nPlayerId": 888888,  "seat": 2, "score": 1200, "bRobot": 1},   # 机器人
    ]
}

# 整个请求体
request_body = {
    "Cmd": "QueryAISingleData",
    "QueryData": query_data  # 可以是 dict，Lambda 会自动处理
}

# ===== 生成 Content-Hmac =====
body_str = json.dumps(request_body, separators=(',', ':'), ensure_ascii=False)
hmac_digest = hmac.new(
    key=SECRET_KEY,
    msg=body_str.encode('utf-8'),
    digestmod=hashlib.sha1
).digest()
content_hmac = base64.b64encode(hmac_digest).decode('utf-8')

# ===== 发送请求 =====
headers = {
    "Content-Type": "application/json",
    "Content-Hmac": content_hmac
}

response = requests.post(LAMBDA_URL, json=request_body, headers=headers)

# ===== 验证响应签名（可选但推荐）=====
received_hmac = response.headers.get("Content-Hmac")
response_body_str = response.text

# 重新计算响应签名
expected_hmac = base64.b64encode(
    hmac.new(
        SECRET_KEY,
        response_body_str.encode('utf-8'),
        hashlib.sha1
    ).digest()
).decode('utf-8')

if received_hmac != expected_hmac:
    raise ValueError("Response HMAC verification failed!")

# ===== 解析结果 =====
if response.status_code == 200:
    result = response.json()
    print("✅ Success:")
    print(json.dumps(result, indent=2, ensure_ascii=False))
else:
    print("❌ Error:", response.status_code, response.text)

✅ Success:
{
  "Res": 1,
  "ResData": {
    "bEffect": 1,
    "handCards": [
      {
        "seat": 0,
        "cards": [
          73,
          23
        ],
        "nPlayerId": 144365
      },
      {
        "seat": 1,
        "cards": [
          42,
          53
        ],
        "nPlayerId": 999999
      },
      {
        "seat": 2,
        "cards": [
          36,
          61
        ],
        "nPlayerId": 888888
      }
    ],
    "commonCards": [
      74,
      17,
      68,
      25,
      50
    ],
    "bPreset": false,
    "inventionType": 0
  },
  "Msg": "Success",
  "Delta": 0.019765615463256836
}


In [ ]:
# 5 * * * * python /mnt/workspace/scripts/game_score2redis.py > /mnt/workspace/logs/game_score2redis_$(date +\%Y\%m\%d).log 2>&1